In [1]:
import os

import torch
import torch.nn.functional as F

import numpy as np
import pandas as pd
from sklearn.metrics import confusion_matrix, classification_report, ConfusionMatrixDisplay
from sklearn.linear_model import LogisticRegressionCV
from sklearn.model_selection import StratifiedKFold
from sklearn.decomposition import PCA
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

from scipy import stats

import matplotlib.pyplot as plt
from matplotlib.colors import LinearSegmentedColormap
import seaborn as sns

import random

from datasets import load_from_disk
from torch.utils.data import DataLoader

from utils.img_encoder import ImgEncoder
from utils.expr_encoder import ExprEncoder
from utils.loss_function import contrastive_loss
from utils.dataset_class import Multi_Modal_Dataset

from utils.config import PROJECT_DIR

import optuna

In [2]:
checkpoint_path = os.path.join(PROJECT_DIR, "checkpoints", "selected", "best.ckpt")
checkpoint = torch.load(checkpoint_path, map_location="cpu")
hparams = checkpoint['hyper_parameters']


img_encoder = ImgEncoder(
    in_dim=1843,
    hidden_dims=hparams["img_hidden_dims"],
    out_dim=128,
    dropout_probs=hparams["img_dropout_probs"],
    batchnorm=True # [*]
)

expr_encoder = ExprEncoder(
    in_dim=2000,
    hidden_dims=hparams["expr_hidden_dims"],
    out_dim=128,
    dropout_probs=hparams["expr_dropout_probs"],
    batchnorm=True # [*]
)


state_dict = checkpoint['state_dict']

img_state_dict = {k.replace('img_encoder.', ''): v for k, v in state_dict.items() if k.startswith('img_encoder.')}
expr_state_dict = {k.replace('expr_encoder.', ''): v for k, v in state_dict.items() if k.startswith('expr_encoder.')}

# Load into models
img_encoder.load_state_dict(img_state_dict)
expr_encoder.load_state_dict(expr_state_dict)

img_encoder.eval()
expr_encoder.eval()

# def (img, expr, pid):
#     return img_encoder(img), expr_encoder(expr), pid
    
print("Encoder weights loaded successfully!")

Encoder weights loaded successfully!


In [3]:
arrow_dataset = load_from_disk(os.path.join(PROJECT_DIR, "data", "dataset"))
all_data = {}
with torch.no_grad():
    for k in ['train', 'valid', 'test']:
        img, ks, expr, pid = next(iter(DataLoader(
            Multi_Modal_Dataset(arrow_dataset[k], k_patches="all"),
            collate_fn=Multi_Modal_Dataset.collate_fn,
            batch_size=len(arrow_dataset[k])
        )))
        all_data[k+'_img'] = img
        all_data[k+'_expr'] = expr
        all_data[k+'_img_emb'] = img_encoder(img)
        all_data[k+'_expr_emb'] = expr_encoder(expr)
        all_data[k+'_ks'] = ks
        all_data[k+'_pid'] = [p[:12] for p in pid]

for x in ['_img', '_expr']:
    for y in ['', '_emb']:
        all_data['dev'+x+y] = torch.cat((all_data['train'+x+y], all_data['valid'+x+y]), dim=0)
all_data['dev_ks'] = all_data['train_ks'] + all_data['valid_ks']
all_data['dev_pid'] = all_data['train_pid'] + all_data['valid_pid']

# sims = F.cosine_similarity(
#     all_data['test_img_emb'].unsqueeze(1), 
#     all_data['dev_img_emb'].unsqueeze(0), 
#     dim=2
# )

# # sanity check!
# for i in range(all_data['test_img_emb'].shape[0]):
#     for j in range(all_data['dev_img_emb'].shape[0]):
#         assert abs(sims[i, j].item() - torch.matmul(
#             F.normalize(all_data['test_img_emb'][i].unsqueeze(0), dim=1), 
#             F.normalize(all_data['dev_img_emb'][j].unsqueeze(0), dim=1).T
#         ).item()) < 1e-6

# sorted_sims = torch.argsort(sims, dim=1, descending=True)

# pid_mapping = {
#     all_data['test_pid'][i]: [all_data['dev_pid'][j] for j in row.tolist()]
#     for i, row in enumerate(sorted_sims)
# }

In [4]:
all_data.keys()

dict_keys(['train_img', 'train_expr', 'train_img_emb', 'train_expr_emb', 'train_ks', 'train_pid', 'valid_img', 'valid_expr', 'valid_img_emb', 'valid_expr_emb', 'valid_ks', 'valid_pid', 'test_img', 'test_expr', 'test_img_emb', 'test_expr_emb', 'test_ks', 'test_pid', 'dev_img', 'dev_img_emb', 'dev_expr', 'dev_expr_emb', 'dev_ks', 'dev_pid'])

In [ ]:
contrastive_loss(z_v=all_data['valid_img_emb'], 
                 ks=all_data['valid_ks'], 
                 z_g=all_data['valid_expr_emb'], 
                 temperature=hparams['temperature'])

tensor(4.2990)

In [13]:
contrastive_loss(z_v=all_data['test_img_emb'], 
                 ks=all_data['test_ks'], 
                 z_g=all_data['test_expr_emb'], 
                 temperature=hparams['temperature'])

tensor(4.3351)